In [3]:
# ============================================================
# Notebook    : 00_data_collection.ipynb
# Description : Collect three datasets for UBI underwriting research
#               - freMTPL2 (HuggingFace)
#               - fremotor2freq9907b (CASdatasets via R script)
#               - car-insurance-data (Kaggle)
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install datasets kaggle pandas


# ============================================================
# 1. Common imports
# ============================================================
import os
import subprocess
import glob
import pandas as pd

os.makedirs("data", exist_ok=True)


# ============================================================
# 2. Dataset 1 — freMTPL2 (HuggingFace: mabilton/fremtpl2)
# ============================================================
from datasets import load_dataset

ds_freq = load_dataset("mabilton/fremtpl2", "freMTPL2freq")
ds_sev  = load_dataset("mabilton/fremtpl2", "freMTPL2sev")

df_freq = ds_freq["train"].to_pandas()
df_sev  = ds_sev["train"].to_pandas()

print("freMTPL2freq shape:", df_freq.shape)
print("freMTPL2sev  shape:", df_sev.shape)
print(df_freq.head(3))

df_freq.to_csv("data/freMTPL2freq.csv", index=False)
df_sev.to_csv("data/freMTPL2sev.csv",  index=False)
print("freMTPL2 saved.")


# ============================================================
# 3. Dataset 2 — fremotor2freq9907b (CASdatasets via R script)
#
#    NOTE: requires get_casdatasets.R in the same folder as this
#    notebook, with the following content:
#
#    Sys.setenv(R_DEFAULT_INTERNET_TIMEOUT = "600")
#    options(timeout = 600)
#    options(download.file.method = "libcurl")
#    if (!requireNamespace("xts", quietly = TRUE)) {
#      install.packages("xts", repos = "https://cloud.r-project.org")
#    }
#    if (!requireNamespace("CASdatasets", quietly = TRUE)) {
#      install.packages(
#        "CASdatasets",
#        repos = "https://dutangc.perso.math.cnrs.fr/RRepository/pub/",
#        type  = "source"
#      )
#    }
#    library(CASdatasets)
#    data(fremotor2freq9907b)
#    write.csv(fremotor2freq9907b, "data/fremotor2freq9907b.csv", row.names = FALSE)
#    cat("fremotor2freq9907b saved.\n")
#
#    R path is auto-detected via glob since each machine may have
#    R installed under a different version folder.
# ============================================================
rscript_candidates = glob.glob(r"C:\Program Files\R\R-*\bin\Rscript.exe")

if not rscript_candidates:
    print("WARNING: Rscript.exe not found under C:\\Program Files\\R\\")
    print("Install R from https://cran.r-project.org/bin/windows/base/")
    print("or set RSCRIPT_PATH manually below.")
    RSCRIPT_PATH = None
else:
    RSCRIPT_PATH = rscript_candidates[0]
    print("Found Rscript at:", RSCRIPT_PATH)

df_motor = None

if RSCRIPT_PATH and os.path.exists("get_casdatasets.R"):
    result = subprocess.run(
        [RSCRIPT_PATH, "get_casdatasets.R"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        cwd=os.getcwd()
    )

    print("STDOUT:")
    print(result.stdout)

    if result.returncode != 0:
        print("STDERR:")
        print(result.stderr)
    else:
        df_motor = pd.read_csv("data/fremotor2freq9907b.csv")
        print("fremotor2freq9907b shape:", df_motor.shape)
        print(df_motor.head(3))
else:
    if not os.path.exists("get_casdatasets.R"):
        print("WARNING: get_casdatasets.R not found in current folder.")
        print("Create it with the R script shown in the comment above.")


# ============================================================
# 4. Dataset 3 — car-insurance-data (Kaggle: sagnik1511)
#
#    NOTE: requires kaggle.json at ~/.kaggle/kaggle.json
# ============================================================
import kaggle

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    "sagnik1511/car-insurance-data",
    path  = "data/car_insurance",
    unzip = True
)

df_car = pd.read_csv("data/car_insurance/Car_Insurance_Claim.csv")

print("car-insurance-data shape:", df_car.shape)
print(df_car.head(3))

df_car.to_csv("data/car_insurance.csv", index=False)
print("car-insurance-data saved.")


# ============================================================
# 5. Summary check
# ============================================================
datasets = {
    "freMTPL2freq"       : df_freq,
    "freMTPL2sev"        : df_sev,
    "car_insurance"      : df_car,
}
if df_motor is not None:
    datasets["fremotor2freq9907b"] = df_motor

print("\n===== Dataset Summary =====")
for name, df in datasets.items():
    print(f"{name:25s} | rows: {df.shape[0]:>7,} | cols: {df.shape[1]:>3}")
print("===========================")

if df_motor is None:
    print("\n⚠ fremotor2freq9907b was NOT loaded.")
    print("  Either R is not installed, or get_casdatasets.R is missing,")
    print("  or the CASdatasets download failed. Check messages above.")

freMTPL2freq shape: (678013, 12)
freMTPL2sev  shape: (26639, 2)
   IDpol  ClaimNb  Exposure  VehPower  VehAge  DrivAge  BonusMalus VehBrand  \
0      1        1      0.10         5       0       55          50      B12   
1      3        1      0.77         5       0       55          50      B12   
2      5        1      0.75         6       2       52          50      B12   

    VehGas Area  Density       Region  
0  Regular    D     1217  Rhone-Alpes  
1  Regular    D     1217  Rhone-Alpes  
2   Diesel    B       54     Picardie  
freMTPL2 saved.
Found Rscript at: C:\Program Files\R\R-4.6.1\bin\Rscript.exe
STDOUT:
package 'zoo' successfully unpacked and MD5 sums checked
package 'xts' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\miy\AppData\Local\Temp\RtmpARuopE\downloaded_packages
fremotor2freq9907b saved.

fremotor2freq9907b shape: (366354, 7)
     IDpol  Year  NbClaim      Expo   Usage    VehType   VehPower
0  PN24816  1999        0 

In [2]:
r_script_content = '''
Sys.setenv(R_DEFAULT_INTERNET_TIMEOUT = "600")
options(timeout = 600)
options(download.file.method = "libcurl")

user_lib <- file.path(Sys.getenv("USERPROFILE"), "R", "win-library", "4.6")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

if (!requireNamespace("xts", quietly = TRUE)) {
  install.packages("xts", repos = "https://cloud.r-project.org", lib = user_lib)
}

if (!requireNamespace("CASdatasets", quietly = TRUE)) {
  install.packages(
    "CASdatasets",
    repos = "https://dutangc.perso.math.cnrs.fr/RRepository/pub/",
    type  = "source",
    lib   = user_lib
  )
}

library(CASdatasets, lib.loc = user_lib)
data(fremotor2freq9907b)
write.csv(fremotor2freq9907b, "data/fremotor2freq9907b.csv", row.names = FALSE)
cat("fremotor2freq9907b saved.\\n")
'''

with open("get_casdatasets.R", "w", encoding="utf-8") as f:
    f.write(r_script_content)

print("get_casdatasets.R updated with user-writable library path.")

get_casdatasets.R updated with user-writable library path.
